# PhishGuard AI — Notebook de modelado (entrega 2)
**Unidad:** Business Process Support · **Proyecto:** clasificación de URLs potencialmente fraudulentas · **Dataset:** PhishVN v3.1.0 · **Autor:** proyecto individual

Este cuaderno utiliza las secciones 0–13 del enunciado `PROJECT_BRIEF_2.md`. La plantilla original `resources/project-model-template.ipynb` no estaba disponible entre los documentos facilitados; mantenemos los títulos y requisitos descritos en el enunciado. Las cifras se obtienen ejecutando el código: no se simulan predicciones.

**Lectura segura:** no abre ni visita URLs sospechosas. El CSV derivado conserva únicamente indicadores numéricos y metadatos de evaluación, no enlaces completos.

## 1. From the brief to an ML task — Del problema a la predicción

| Pregunta | Aplicación en PhishGuard AI |
|---|---|
| Decisión | Qué enlaces revisar primero y cuándo pedir verificación adicional |
| Tipo | Clasificación binaria supervisada |
| Objetivo | `label`: 1 = URL en fuentes de fraude/phishing; 0 = URL catalogada como legítima |
| Granularidad | Una predicción por URL, no por usuario, mensaje o incidente |
| Momento | Cuando el SOC recibe una URL, antes de cualquier investigación |
| Variables disponibles | Solo atributos de la cadena URL calculables en ese instante |
| Regla de referencia | Revisar la URL si la longitud es ≥ 30 caracteres (regla hipotética, no operativa) |
| Baseline ingenuo | Predecir la clase mayoritaria del entrenamiento |
| Métrica principal | Recall sujeto a tasa de falsos positivos (FPR) ≤ 5 % en validación |
| Traducción empresarial | Alertas fraudulentas detectadas y falsos avisos por cada 1.000 URLs evaluadas |

**Cambio justificado frente al Brief:** el análisis exploratorio mostró que la mediana de longitud de positivas era menor, no mayor: H1 se revisa, no se oculta. La etiqueta positiva cubre fraude más amplio que el phishing de credenciales. Se mantiene la decisión del SOC y el dataset. Las metas propuestas de recall ≥ 90 % y FPR ≤ 5 % son requisitos aspiracionales; no se deben declarar logradas sin medirlas. La regla de 30 caracteres es hipotética, no una política real del SOC.

*Qué significa para el responsable:* elegimos los errores que importan a un SOC, no la exactitud aislada.

## 2. Data and quality — Datos y limpieza
Se utiliza `project/data/phishvn_model.csv`, preparado desde el ZIP original combinando las **21 características léxicas** de `vn_compphish.csv` con `source`, `collected_at` y `domain` de `dataset_url.csv` mediante la clave `url_norm` (no por orden de filas). Se eliminaron de la copia los enlaces completos, `is_https` (artefacto documentado), `is_ip` (constante) y atributos que podrían revelar etiquetas. El CSV es derivado y conserva licencia CC BY 4.0. En el CSV derivado hay **7 filas con indicadores y metadatos idénticos**, aunque el archivo original tenía 53.116 URLs diferentes: no se borran porque representarían URLs distintas y alteraría los splits. Las fechas ausentes afectan al análisis temporal, no a la predicción.

En todos los modelos, las imputaciones y transformaciones se ajustan **dentro del pipeline y exclusivamente sobre entrenamiento**. Las fechas son para auditoría, no variables predictivas.

In [1]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import joblib
import sklearn, plotly
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
SEED = 42
BASE = Path.cwd()
if not (BASE / 'data/phishvn_model.csv').exists() and (BASE / 'project/data/phishvn_model.csv').exists():
    BASE = BASE / 'project'
DATA = BASE / 'data/phishvn_model.csv'
assert DATA.exists(), f'Falta el archivo derivado requerido: {DATA}'
df = pd.read_csv(DATA, low_memory=False)
print('Datos cargados:', df.shape, '| archivo MB:', round(DATA.stat().st_size/1e6, 2))
print('Versiones:', 'pandas', pd.__version__, '| sklearn', sklearn.__version__, '| plotly', plotly.__version__)

Datos cargados: (53116, 26) | archivo MB: 7.39
Versiones: pandas 3.0.3 | sklearn 1.8.0 | plotly 7.1.0


In [2]:
numeric_all = ['url_len','dom_len','tld_len','subdom_cnt','letter_cnt','digit_cnt',
    'special_cnt','eq_cnt','qm_cnt','amp_cnt','dot_cnt','dash_cnt','under_cnt',
    'letter_ratio','digit_ratio','spec_ratio','slash_cnt','entropy','path_len','query_len']
required = set(numeric_all + ['label','split','tier','source','domain','collected_at'])
assert required <= set(df.columns), f'Columnas faltantes: {required-set(df.columns)}'
assert df['label'].isin([0,1]).all() and len(df) == 53116
assert df['split'].value_counts().to_dict() == {'train':36909,'val':7266,'test':8941}
print('Clases:', df['label'].value_counts().sort_index().to_dict())
print('Nulos por atributo (mayores):')
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame('nulos'))
print('Filas exactas duplicadas:', df.duplicated().sum())
print('Duplicados de combinación (dominio, split, métrica longitud):',df.duplicated(['domain','split','url_len']).sum(), '(no implica URLs duplicadas)')
# Fecha: dos formatos documentados, sin mezclar día y mes.
date_es = pd.to_datetime(df['collected_at'], format='%d/%m/%Y', errors='coerce')
date_iso = pd.to_datetime(df['collected_at'], format='%Y-%m-%d', errors='coerce')
df['fecha'] = date_es.fillna(date_iso)
print('Fechas válidas:',int(df['fecha'].notna().sum()),'de',len(df))
print('Nulos en variables numéricas:',int(df[numeric_all].isna().sum().sum()))

Clases: {0: 16410, 1: 36706}
Nulos por atributo (mayores):


,nulos
collected_at,31455
url_len,0
tld_len,0
subdom_cnt,0
letter_cnt,0
digit_cnt,0
special_cnt,0
eq_cnt,0
qm_cnt,0
dom_len,0


Filas exactas duplicadas: 7
Duplicados de combinación (dominio, split, métrica longitud): 24 (no implica URLs duplicadas)
Fechas válidas: 21661 de 53116
Nulos en variables numéricas: 0


*Qué significa para el responsable:* los nulos y la procedencia limitan las conclusiones. La limpieza queda cuantificada.

## 3. What the exploration showed — Hipótesis y gráficos
Hipótesis procedentes de decisiones del SOC: **H1** pregunta si señales simples (longitud y puntos) permiten priorizar; **H2** pregunta si combinar rasgos aumenta la utilidad frente a una regla; **H3** pregunta si una misma regla perjudica más a unas fuentes legítimas que a otras. H2 y H3 solo se podrán cerrar cuando existan predicciones. Las características derivadas usan la URL sin protocolo: aquí la mediana es **14 caracteres en positivos y 16 en benignos**, distinta de las cifras del CSV principal (**15 y 17**). Los gráficos describen **este corpus** solamente.

Se muestran distribución de etiquetas, diferencias de longitud y fechas disponibles. Los gráficos temporales no equiparan fechas de certificación y detección.

In [3]:
def chart(fig, name):
    fig.update_layout(template='plotly_white', width=780, height=430, margin=dict(l=55,r=30,t=65,b=55))
    fig.show()
    # Versión interactiva independiente, útil si GitHub no renderiza Plotly en el .ipynb.
    dest = BASE / 'figures' / (name+'.html')
    dest.parent.mkdir(exist_ok=True)
    fig.write_html(dest, include_plotlyjs='directory', full_html=True)

counts=df['label'].map({0:'Legítima (0)',1:'Positiva (1)'}).value_counts().reset_index()
counts.columns=['clase','urls']
chart(px.bar(counts,x='clase',y='urls',color='clase',text='urls',
             title='Figura 1. URLs por etiqueta',labels={'clase':'Clase','urls':'URLs (número)'}),'01_clases')

![Figura 1 — vista para GitHub](figures/01_clases.png)

In [4]:
h1=df.groupby('label')[['url_len','dot_cnt']].agg(['count','median','mean']).round(2)
display(h1)
plot_df=df[['label','url_len']].copy()
plot_df['etiqueta']=plot_df.label.map({0:'Legítima',1:'Positiva'})
lo,hi=plot_df.url_len.quantile([0.01,0.99]); plot_df=plot_df[plot_df.url_len.between(lo,hi)]
chart(px.box(plot_df,x='etiqueta',y='url_len',color='etiqueta',points=False,
             title='Figura 2. Longitud de URL por clase (p1–p99)',
             labels={'etiqueta':'Etiqueta','url_len':'Longitud (caracteres)'}),'02_longitudes')

url_len               dot_cnt             
        count median   mean   count median  mean
label                                           
0       16410   16.0  18.19   16410    2.0  2.02
1       36706   14.0  15.67   36706    2.0  1.52

![Vista estática de la Figura 2 para GitHub](figures/02_longitudes.png)

In [5]:
temporal=df.loc[df.label.eq(1) & df.fecha.notna(), ['fecha']].copy()
monthly=(temporal.assign(mes=temporal.fecha.dt.to_period('M').dt.to_timestamp())
         .groupby('mes').size().rename('urls').reset_index())
chart(px.line(monthly,x='mes',y='urls',markers=True,
              title='Figura 3. Registros positivos fechados por mes (no incidencia real)',
              labels={'mes':'Mes de registro','urls':'URLs registradas (número)'}),'03_fechas')
print('Registros positivos fechados:',len(temporal),'| desde:',monthly.mes.min().date(),'hasta:',monthly.mes.max().date())

Registros positivos fechados: 19635 | desde: 2020-02-01 hasta: 2025-02-01


![Vista estática de la Figura 3 para GitHub](figures/03_fechas.png)

| Hipótesis | Estado en la exploración | Evidencia / siguiente prueba |
|---|---|---|
| H1: positivas más largas y con más puntos | **Refutada en longitud (descriptivo)** | Comparar medianas; no asumir que los puntos discriminan por sí solos |
| H2: varias características mejoran la regla | **Aún no evaluable aquí** | Comparar F1, recall y FPR de reglas y modelos en validación |
| H3: cambia la falsa alarma según fuente legítima | **Aún no evaluable aquí** | Agrupar falsos positivos por `source` tras predecir |

*Qué significa para el responsable:* no conviene justificar alertas solo porque una dirección web sea larga; las otras dos hipótesis necesitan mediciones reales del clasificador.

## 4. Features — Variables y control de leakage

| Variables | Cómo se obtienen / por qué podrían ayudar | Uso |
|---|---|---|
| `url_len`, `dom_len`, `path_len`, `query_len` | Longitud de URL, dominio, ruta y consulta | Entrada disponible al recibir URL |
| `letter_cnt`, `digit_cnt`, `special_cnt`, ratios | Conteos y proporciones de letras, dígitos y símbolos | Entrada |
| `dot_cnt`, `dash_cnt`, `under_cnt`, `slash_cnt`, `eq_cnt`, `qm_cnt`, `amp_cnt` | Frecuencias de separadores | Entrada |
| `subdom_cnt`, `tld_len` | Subdominios y tamaño de extensión | Entrada |
| `entropy` | Entropía de Shannon según el featurizador de PhishVN | Entrada |
| `source`, `tier`, `split`, `collected_at`, `domain` | Procedencia, confianza, partición, fecha y grupo | **Solo auditoría/split; prohibidas como entrada** |
| `label`, `is_https`, `is_ip`, URL literal | Etiqueta, artefacto, columna constante e identificador | **Excluidas** |

**Comprobación del momento de predicción:** las 20 características seleccionadas se pueden calcular a partir del texto de la URL antes de consultar su reputación. Sin embargo, este notebook recibe esos 20 valores ya extraídos en el CSV de PhishVN. Para desplegarlo a partir de una URL cruda, el bloque 4 necesitará un featurizador idéntico y probado; no afirmamos que el archivo `.joblib` acepte directamente una URL como cadena. La imputation y el escalado se realizan solamente en el `Pipeline`.

In [6]:
excluded = sorted(set(df.columns)-set(numeric_all))
print('Columnas usadas para predecir:',numeric_all)
print('Columnas excluidas:',excluded)
assert all(x not in numeric_all for x in ['label','source','tier','split','fecha','domain','collected_at','is_https'])
print('Disponibilidad de cada variable (porcentaje):')
display((100*df[numeric_all].notna().mean()).round(1).to_frame('Disponible (%)').T)

Columnas usadas para predecir: ['url_len', 'dom_len', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'slash_cnt', 'entropy', 'path_len', 'query_len']
Columnas excluidas: ['collected_at', 'domain', 'fecha', 'label', 'source', 'split', 'tier']
Disponibilidad de cada variable (porcentaje):


,url_len,dom_len,tld_len,subdom_cnt,letter_cnt,digit_cnt,special_cnt,eq_cnt,qm_cnt,amp_cnt,dot_cnt,dash_cnt,under_cnt,letter_ratio,digit_ratio,spec_ratio,slash_cnt,entropy,path_len,query_len
Disponible (%),100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


*Qué significa para el responsable:* elegimos variables disponibles al recibir el enlace. La procedencia no condiciona directamente la predicción.

## 5. Split and baseline — Particiones y reglas simples
Se conservan los splits oficiales: **train 36.909**, **validation 7.266**, **test 8.941**. Fueron diseñados por los autores para separar dominios registrados y secuenciar parte de los registros fechados. No son una separación cronológica perfecta para todos los ejemplos: **59,2 % no tiene fecha**, y el origen de datos modifica la composición de las clases. Inventar fechas o eliminar todos los negativos sin fecha falsearía el problema. Antes de entrenar comprobamos que un `domain` no aparece en varias particiones.

Dos referencias: (a) siempre predecir la clase mayoritaria del entrenamiento; (b) regla hipotética `url_len >= 30` para revisión. El corte se fija a priori según la intuición original de H1, no se optimiza sobre validation. Las puntuaciones de las reglas se calculan solo sobre validation hasta la evaluación final.

In [7]:
train=df.loc[df.split.eq('train')].copy(); val=df.loc[df.split.eq('val')].copy()
# test se separa y NO se consulta hasta la evaluación final en la sección 8.
train_domains=set(train.domain); val_domains=set(val.domain); test_domains=set(df.loc[df.split.eq('test'),'domain'])
print('Solapamiento de dominios exactos: train/val',len(train_domains&val_domains),
      'train/test',len(train_domains&test_domains),'val/test',len(val_domains&test_domains))
assert not(train_domains&val_domains or train_domains&test_domains or val_domains&test_domains)
X_train,y_train = train[numeric_all], train.label
X_val,y_val = val[numeric_all], val.label
majority = int(y_train.mode().iloc[0])
def row_metrics(y, yp, tag):
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return {'modelo':tag,'precision':precision_score(y,yp,zero_division=0),
            'recall':recall_score(y,yp,zero_division=0),'f1':f1_score(y,yp,zero_division=0),
            'fpr':fp/(fp+tn) if fp+tn else 0,'fn':int(fn),'fp':int(fp),'tp':int(tp),'tn':int(tn)}
base_major=np.full(len(val),majority,dtype=int)
base_rule=(val['url_len'].values>=30).astype(int)
base_table=pd.DataFrame([row_metrics(y_val,base_major,'Mayoritaria'),
                         row_metrics(y_val,base_rule,'Regla longitud ≥ 30')])
display(base_table.round(4))
print('Proporción de positivas por partición: ',df.groupby('split').label.agg(['count','mean']).round(3))

Solapamiento de dominios exactos: train/val 0 train/test 0 val/test 0


,modelo,precision,recall,f1,fpr,fn,fp,tp,tn
0,Mayoritaria,0.7852,1.0000,0.8797,1.0000,0,1561,5705,0
1,Regla longitud ≥ 30,0.9673,0.0259,0.0505,0.0032,5557,5,148,1556


Proporción de positivas por partición:         count   mean
split              
test    8941  0.570
train  36909  0.702
val     7266  0.785


*Qué significa para el responsable:* hay que superar la regla simple; la prevalencia del dataset no representa al SOC.

## 6. Candidate models — Modelos candidatos
Comparamos **regresión logística** (explicable y rápida, con escalado) y **Random Forest** (capta interacciones). Ambas son pipelines de scikit-learn que imputan dentro del entrenamiento. Primero medimos en validation con umbral 0,50 sin ajustar y añadimos los baselines. La elección posterior valorará la restricción operativa FPR ≤ 5 %, no solo un F1 alto.

In [8]:
logistic=Pipeline([('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler()),('classifier',LogisticRegression(max_iter=800,random_state=SEED))])
forest=Pipeline([('imputer',SimpleImputer(strategy='median')),
    ('classifier',RandomForestClassifier(n_estimators=90,max_depth=12,min_samples_leaf=5,
                                        random_state=SEED,n_jobs=-1))])
models={'Regresión logística':logistic,'Random Forest':forest}
prob_val={}; model_rows=[]
for name,pipe in models.items():
    t0=time.perf_counter();pipe.fit(X_train,y_train)
    probs=pipe.predict_proba(X_val)[:,1];prob_val[name]=probs
    r=row_metrics(y_val,(probs>=.5).astype(int),name);r['auc_roc']=roc_auc_score(y_val,probs)
    model_rows.append(r)
    print(name,'entrenado en',round(time.perf_counter()-t0,2),'segundos')
comparison=pd.concat([base_table,pd.DataFrame(model_rows)],ignore_index=True)
display(comparison.round(4))

Regresión logística entrenado en 0.22 segundos
Random Forest entrenado en 0.62 segundos


,modelo,precision,recall,f1,fpr,fn,fp,tp,tn,auc_roc
0,Mayoritaria,0.7852,1.0000,0.8797,1.0000,0,1561,5705,0,NaN
1,Regla longitud ≥ 30,0.9673,0.0259,0.0505,0.0032,5557,5,148,1556,NaN
2,Regresión logística,0.8839,0.9621,0.9214,0.4619,216,721,5489,840,0.8971
3,Random Forest,0.8924,0.9434,0.9172,0.4158,323,649,5382,912,0.9156


In [10]:
# Ablación: misma familia de modelo, solo dos rasgos frente a los veinte del experimento.
# Ejemplo de decisión de diseño que puede resultar peor: se muestra, no se oculta.
small_features=['url_len','dot_cnt']
small_forest=Pipeline([('imputer',SimpleImputer(strategy='median')),
    ('classifier',RandomForestClassifier(n_estimators=90,max_depth=12,min_samples_leaf=5,
                                        random_state=SEED,n_jobs=-1))])
small_forest.fit(train[small_features],y_train)
small_pred=small_forest.predict(val[small_features])
full_pred=models['Random Forest'].predict(X_val)
ablation_table=pd.DataFrame([row_metrics(y_val,small_pred,'Solo longitud y puntos'),
                             row_metrics(y_val,full_pred,'20 características')])
display(ablation_table[['modelo','f1','recall','fpr']].round(4))


,modelo,f1,recall,fpr
0,Solo longitud y puntos,0.8919,0.9865,0.8245
1,20 características,0.9172,0.9434,0.4158


In [9]:
long_comp=comparison.melt(id_vars='modelo',value_vars=['recall','fpr'],var_name='métrica',value_name='proporción')
chart(px.bar(long_comp,x='modelo',y='proporción',color='métrica',barmode='group',
             title='Figura 4. Recall y FPR con umbral 0,50 en validación',
             labels={'modelo':'Método','proporción':'Proporción (0–1)','métrica':'Indicador'}),'04_modelos')

![Vista estática de la Figura 4 para GitHub](figures/04_modelos.png)

*Qué significa para el responsable:* F1 alto no basta si produce demasiadas falsas alarmas.

## 7. Metric and cost of errors — Umbral y costes
**Falso negativo (FN):** URL fraudulenta no señalada; riesgo de no investigar. **Falso positivo (FP):** URL legítima enviada a revisión; genera esfuerzo y posibles bloqueos injustificados. Nuestra regla operativa inicial exige **FPR ≤ 5 %** en validación, y dentro de esa restricción busca el mayor recall. Sin costes reales del SOC, no estimamos euros.

El umbral se elige **solo con validation** entre valores predefinidos. Se comparan ambos modelos bajo igual límite. Si la meta de recall ≥ 90 % no se alcanza, la recomendación no será bloqueo automático.

In [11]:
threshold_grid=np.round(np.linspace(0.05,0.995,190),4)
threshold_rows=[]
for name,pr in prob_val.items():
    for threshold in threshold_grid:
        metrics=row_metrics(y_val,(pr>=threshold).astype(int),name)
        threshold_rows.append({'modelo':name,'umbral':threshold,**{k:metrics[k] for k in ['recall','fpr','precision','f1','fp','fn']}})
thresh_table=pd.DataFrame(threshold_rows)
feasible=thresh_table[thresh_table.fpr<=0.05].copy()
assert not feasible.empty,'Ningún umbral cumple el límite de FPR'
# Priorizamos recall y después simplicidad/precision solo cuando hay empates.
selected_row=feasible.sort_values(['recall','fpr','precision'],ascending=[False,True,False]).iloc[0]
selected_name=str(selected_row['modelo']); chosen_threshold=float(selected_row['umbral'])
print('Elección previa al test:',selected_name,'| umbral:',chosen_threshold,
      '| recall_validación:',round(float(selected_row.recall),4),
      '| FPR_validación:',round(float(selected_row.fpr),4))
display(feasible.sort_values(['recall','fpr'],ascending=[False,True]).groupby('modelo').head(1).round(4))

Elección previa al test: Random Forest | umbral: 0.81 | recall_validación: 0.7087 | FPR_validación: 0.05


,modelo,umbral,recall,fpr,precision,f1,fp,fn
342,Random Forest,0.81,0.7087,0.0500,0.9811,0.8229,78,1662
172,Regresión logística,0.91,0.6103,0.0493,0.9784,0.7517,77,2223


In [12]:
threshold_curve=thresh_table[thresh_table.modelo.eq(selected_name)].copy()
chart(px.line(threshold_curve,x='fpr',y='recall',hover_data=['umbral'],
              title='Figura 5. Compromiso recall vs falsas alarmas (validación)',
              labels={'fpr':'Tasa de falsos positivos (0–1)','recall':'Recall (0–1)'}),'05_umbral')
val_pred=(prob_val[selected_name]>=chosen_threshold).astype(int)
val_cm=confusion_matrix(y_val,val_pred,labels=[0,1])
chart(px.imshow(val_cm,text_auto=True,aspect='auto',color_continuous_scale='Blues',
                x=['Predicha legítima','Predicha positiva'],y=['Real legítima','Real positiva'],
                title='Figura 6. Matriz de confusión en validación (número de URLs)',
                labels={'x':'Predicción','y':'Etiqueta real','color':'URLs'}),'06_confusion_validacion')

![Figura 6 — vista para GitHub](figures/06_confusion_validacion.png)

![Vista estática de la Figura 5 para GitHub](figures/05_umbral.png)

*Qué significa para el responsable:* el umbral equilibra amenazas perdidas y revisiones; debe acordarse con el SOC.

## 8. Validation and tuning — Validación cruzada, ajuste y prueba final
**Validación cruzada:** `GroupKFold` de tres pliegues en entrenamiento, agrupado por dominio, evita que un mismo dominio exacto contamine pliegues distintos. No hacemos `TimeSeriesSplit`: muchas fechas faltan. **Falta una prueba estrictamente temporal** con datos futuros. Se comparan media y dispersión entre pliegues para detectar inestabilidad. Ajustamos hiperparámetros solo sobre entrenamiento, volvemos a calibrar el umbral en validation y por último realizamos **una única evaluación final en test**, sin volver a tomar decisiones tras verla.

Las fuentes y dominios podrían introducir optimismo; los splits oficiales agrupan dominios registrados.

In [13]:
groups=train.domain.fillna('sin_dominio').astype(str)
cv=GroupKFold(n_splits=3)
selected_pipeline=models[selected_name]
cv_scores=cross_val_score(selected_pipeline,X_train,y_train,groups=groups,
                          cv=cv,scoring='f1',n_jobs=1)
print('F1 por pliegue (0,50):',np.round(cv_scores,4),
      '| media:',round(float(cv_scores.mean()),4),'| desviación:',round(float(cv_scores.std()),4))
if selected_name=='Random Forest':
    search_grid={'classifier__max_depth':[9,12], 'classifier__min_samples_leaf':[5,10]}
else:
    search_grid={'classifier__C':[0.3,1.0,3.0]}
search=GridSearchCV(selected_pipeline,param_grid=search_grid,scoring='f1',cv=cv,
                    n_jobs=1,refit=True,return_train_score=True)
search.fit(X_train,y_train,groups=groups)
print('Mejores parámetros:',search.best_params_,'| CV F1:',round(search.best_score_,4))
best_pipe=search.best_estimator_
tune_prob=best_pipe.predict_proba(X_val)[:,1]
tune_rows=[]
for tau in threshold_grid:
    m=row_metrics(y_val,(tune_prob>=tau).astype(int),'ajustado')
    tune_rows.append((float(tau),m['fpr'],m['recall'],m['precision']))
tunet=pd.DataFrame(tune_rows,columns=['tau','fpr','recall','precision'])
tunet=tunet[tunet.fpr<=.05].sort_values(['recall','fpr','precision'],ascending=[False,True,False])
assert not tunet.empty
final_tau=float(tunet.iloc[0].tau)
print('Umbral calibrado exclusivamente en validación:',final_tau,
      '| recall:',round(float(tunet.iloc[0].recall),4),'| fpr:',round(float(tunet.iloc[0].fpr),4))
cv_results=pd.DataFrame(search.cv_results_)[['params','mean_train_score','mean_test_score','std_test_score']]
display(cv_results.round(4))

F1 por pliegue (0,50): [0.9229 0.9222 0.9232] | media: 0.9228 | desviación: 0.0004
Mejores parámetros: {'classifier__max_depth': 12, 'classifier__min_samples_leaf': 5} | CV F1: 0.9228
Umbral calibrado exclusivamente en validación: 0.81 | recall: 0.7087 | fpr: 0.05


,params,mean_train_score,mean_test_score,std_test_score
0,"{'classifier__max_depth': 9, 'classifier__min_...",0.9244,0.9227,0.0006
1,"{'classifier__max_depth': 9, 'classifier__min_...",0.9235,0.9222,0.0007
2,"{'classifier__max_depth': 12, 'classifier__min...",0.9250,0.9228,0.0004
3,"{'classifier__max_depth': 12, 'classifier__min...",0.9239,0.9224,0.0005


In [14]:
# EVALUACIÓN FINAL: se abre el conjunto test por primera vez aquí y solo se evalúa una vez.
test=df.loc[df.split.eq('test')].copy(); X_test=test[numeric_all]; y_test=test.label
test_prob=best_pipe.predict_proba(X_test)[:,1]
test_pred=(test_prob>=final_tau).astype(int)
final_metrics=row_metrics(y_test,test_pred,'Modelo final')
final_metrics['auc_roc']=roc_auc_score(y_test,test_prob)
display(pd.DataFrame([final_metrics]).round(4))
print('Resultado final aislado: recall',round(final_metrics['recall'],4),
      '| FPR',round(final_metrics['fpr'],4), '| precision',round(final_metrics['precision'],4),
      '| F1',round(final_metrics['f1'],4))
print('Metas simultáneas recall≥90% / FPR≤5% en test:',final_metrics['recall']>=.9 and final_metrics['fpr']<=.05)

,modelo,precision,recall,f1,fpr,fn,fp,tp,tn,auc_roc
0,Modelo final,0.9706,0.6799,0.7997,0.0273,1631,105,3465,3740,0.9635


Resultado final aislado: recall 0.6799 | FPR 0.0273 | precision 0.9706 | F1 0.7997
Metas simultáneas recall≥90% / FPR≤5% en test: False


In [15]:
cm=confusion_matrix(y_test,test_pred,labels=[0,1])
chart(px.imshow(cm,text_auto=True,aspect='auto',color_continuous_scale='Blues',
                x=['Predicha legítima','Predicha positiva'],y=['Real legítima','Real positiva'],
                title='Figura 7. Matriz de confusión final: test aislado',
                labels={'x':'Predicción','y':'Etiqueta real','color':'URLs'}),'07_confusion_test')

![Vista estática de la Figura 7 para GitHub](figures/07_confusion_test.png)

*Qué significa para el responsable:* los pliegues muestran estabilidad. El test se utiliza una vez, sin ajustes posteriores.

## 9. Interpretation — Qué variables pesan y ejemplos
Calculamos importancia por permutación **sobre una muestra fija de validación**, sin utilizar test para elegir características. Permutar una característica indica cuánto empeora F1 al alterar esa variable; rasgos correlacionados pueden repartirse la importancia. Mostramos tres casos de test sin publicar enlaces: un acierto positivo, una falsa alarma y un fallo de detección, si existen. Las explicaciones individuales son descriptivas: la puntuación y algunas variables de entrada, no una causa demostrada.

In [16]:
rng=np.random.default_rng(SEED)
ix=rng.choice(len(val),size=min(1500,len(val)),replace=False)
perm=permutation_importance(best_pipe,X_val.iloc[ix],y_val.iloc[ix],
                            n_repeats=3,random_state=SEED,scoring='f1',n_jobs=1)
importance=pd.DataFrame({'variable':numeric_all,'caida_f1':perm.importances_mean,
                         'desviacion':perm.importances_std}).sort_values('caida_f1',ascending=False)
display(importance.head(10).round(4))
chart(px.bar(importance.head(10).sort_values('caida_f1'),x='caida_f1',y='variable',orientation='h',
             title='Figura 8. Importancia por permutación (validación)',
             labels={'caida_f1':'Caída media de F1 al permutar','variable':'Variable'}),'08_importancia')

,variable,caida_f1,desviacion
3,subdom_cnt,0.0537,0.0039
2,tld_len,0.0379,0.0026
10,dot_cnt,0.0148,0.0013
6,special_cnt,0.0029,0.0006
5,digit_cnt,0.0021,0.0003
17,entropy,0.0009,0.0002
15,spec_ratio,0.0001,0.0004
0,url_len,0.0000,0.0005
1,dom_len,0.0000,0.0002
8,qm_cnt,0.0000,0.0000


![Vista estática de la Figura 8 para GitHub](figures/08_importancia.png)

In [17]:
cases=test[['label','tier','source']].copy().reset_index(drop=True)
cases['pred']=test_pred; cases['probabilidad']=test_prob
cases['tipo']=np.select([(cases.label.eq(1)&cases.pred.eq(1)),
    (cases.label.eq(0)&cases.pred.eq(1)),(cases.label.eq(1)&cases.pred.eq(0))],
    ['Acierto positivo','Falsa alarma','Falso negativo'],default='Acierto negativo')
chosen_idx=[]
for typ in ['Acierto positivo','Falsa alarma','Falso negativo']:
    inds=cases.index[cases.tipo.eq(typ)]
    if len(inds):chosen_idx.append(int(inds[0]))
examples=cases.iloc[chosen_idx].copy()
for feat_name in ['url_len','entropy','digit_ratio']:
    examples[feat_name]=X_test.iloc[chosen_idx][feat_name].to_numpy()
display(examples.reset_index(drop=True).round(4))

,label,tier,source,pred,probabilidad,tipo,url_len,entropy,digit_ratio
0,1,silver,tinnhiemmang,1,0.9246,Acierto positivo,23,3.8820,0.0
1,0,gold,tinnhiem_web,1,0.9716,Falsa alarma,13,2.7193,0.0
2,1,silver,tinnhiemmang,0,0.6979,Falso negativo,17,3.3816,0.0


*Qué significa para el responsable:* un indicador estadístico no prueba que una web sea fraudulenta.

## 10. Fairness, ethics and risks — Revisión por grupos
Evaluamos la FPR entre **fuentes de URLs benignas** y el recall entre niveles de confianza positivos. `source` y `tier` no entran en el entrenamiento. Estos segmentos indican **cobertura y condiciones de recopilación**, no grupos protegidos de personas: las diferencias encontradas no prueban discriminación individual. En producción debemos vigilar también idioma, país y cambios de fuente, que no pueden evaluarse aquí con representatividad garantizada.

Riesgos: falsos negativos (amenazas sin investigar), falsos positivos (sitios legítimos castigados), sesgo de procedencia, etiquetas inciertas y exposición innecesaria de URLs maliciosas. Mitigaciones: revisión humana; listas de excepciones auditadas; umbrales supervisados; registro mínimo de metadatos; no visitar enlaces; pruebas de deriva y recalibración antes de desplegar. Las URL pueden contener identificadores o tokens personales, por lo que el CSV derivado público excluye el texto completo.

In [19]:
audit=test[['source','tier','label']].copy().reset_index(drop=True)
audit['pred']=test_pred
ben=audit[audit.label.eq(0)].groupby('source').agg(n=('pred','size'),
                        falsas_alarmas=('pred','sum'))
ben['fpr']=ben.falsas_alarmas/ben.n
pos=audit[audit.label.eq(1)].groupby('tier').agg(n=('pred','size'),detectadas=('pred','sum'))
pos['recall']=pos.detectadas/pos.n
print('Errores en fuentes benignas — Test:')
display(ben.round(4))
print('Detección por confianza en positivos — Test:')
display(pos.round(4))
chart(px.bar(ben.reset_index(),x='source',y='fpr',text='n',
             title='Figura 9. Falsas alarmas por fuente benigna (test)',
             labels={'source':'Fuente benigna','fpr':'Tasa de falsos positivos (0–1)','n':'URLs evaluadas'}),'09_auditoria_fuentes')

Errores en fuentes benignas — Test:


,n,falsas_alarmas,fpr
source,,,
tinnhiem_org,358,9,0.0251
tinnhiem_web,2246,21,0.0093
tranco,888,74,0.0833
tranco_vn,353,1,0.0028


Detección por confianza en positivos — Test:


,n,detectadas,recall
tier,,,
bronze,4609,3245,0.7041
gold,69,34,0.4928
silver,418,186,0.4450


![Vista estática de la Figura 9 para GitHub](figures/09_auditoria_fuentes.png)

*Qué significa para el responsable:* evaluar los errores por fuente evita confiar solamente en el promedio global.

**Cierre de las hipótesis:** H1 se **refuta en la dirección de longitud** con evidencia descriptiva; el número de puntos presenta mediana 2 en ambos grupos. H2 queda **apoyada frente a la regla hipotética evaluada**, no frente a procesos reales que no conocemos. H3 queda **apoyada en esta muestra**: los grupos benignos presentan falsas alarmas desiguales. Estas conclusiones se limitan al dataset PhishVN; una evaluación externa podría cambiarlas.

*Qué significa para el responsable:* la explicación de las hipótesis viene de comparar observaciones y predicciones, no de asumir que las ideas iniciales eran correctas.

## 11. Recommendation — Recomendación para el SOC
En test, el modelo detectó 3.465 de 5.096 positivos y marcó erróneamente 105 de 3.845 legítimas. El **recall = 68,0 %** queda por debajo del 90 % propuesto aunque la **FPR = 2,73 %** queda dentro del límite. **H2 se apoya frente a la regla hipotética**, pero esto no valida su uso real. **H3 se observa** en test: la FPR varía entre fuentes benignas (aprox. **0,28 % en Tranco VN y 8,33 % en Tranco global**), por lo que requiere vigilancia de procedencia. Recomendamos según resultados, no intenciones. **Usar como priorizador en un piloto supervisado** es más defendible que bloquear automáticamente. El equipo debería comprobar su volumen de alertas, coste de revisión y tasa de ataques realmente presentes en su entorno antes de fijar un umbral de producción. La comparación con la regla hipotética es válida como ejercicio técnico, **no como ahorro económico demostrado**.

El bloque 4 debe añadir un featurizador consistente para URLs nuevas, interfaz, registros mínimos, monitorización de deriva y revisión periódica por fuentes. Si no se cumplen simultáneamente las metas de recall y FPR, la recomendación es **no desplegar decisiones automáticas**. No calculamos ahorros monetarios sin costes medidos.

In [20]:
print('RESULTADO EN TEST — POR CADA 1.000 URLs DE ESTA MUESTRA (NO DEL SOC REAL):')
scale=1000/len(y_test)
print('Detectadas positivas:',round(final_metrics['tp']*scale,1),
      '| falsas alarmas:',round(final_metrics['fp']*scale,1),
      '| amenazas no señaladas:',round(final_metrics['fn']*scale,1))
print('Comparación con regla longitud>=30 usando el MISMO test (solo como referencia ex post):')
rule_test=row_metrics(y_test,(test.url_len.values>=30).astype(int),'Regla 30')
print('Regla: recall',round(rule_test['recall'],4),'FPR',round(rule_test['fpr'],4),
      '| modelo final:',round(final_metrics['recall'],4),round(final_metrics['fpr'],4))
print('¿Es apropiado bloquear automáticamente según las metas propuestas?:',
      'No' if not (final_metrics['recall']>=.9 and final_metrics['fpr']<=.05) else
      'Aún no: falta validación externa y autorización humana')

RESULTADO EN TEST — POR CADA 1.000 URLs DE ESTA MUESTRA (NO DEL SOC REAL):
Detectadas positivas: 387.5 | falsas alarmas: 11.7 | amenazas no señaladas: 182.4
Comparación con regla longitud>=30 usando el MISMO test (solo como referencia ex post):
Regla: recall 0.0228 FPR 0.2447 | modelo final: 0.6799 0.0273
¿Es apropiado bloquear automáticamente según las metas propuestas?: No


*Qué significa para el responsable:* sin datos propios no podemos prometer ahorro ni menos incidentes.

## 12. Saving the model — Pipeline guardado y recargado
Se guarda **un único pipeline** de scikit-learn en `project/model.joblib` y el umbral de decisión en un JSON pequeño. El pipeline acepta **20 columnas numéricas** y produce probabilidades; el featurizador para URLs nuevas queda para el bloque 4. El fichero se vuelve a abrir y se realiza una predicción comprobable. Los archivos `joblib` solo deben cargarse de fuentes confiables.

In [21]:
model_file=BASE/'model.joblib';threshold_file=BASE/'decision_threshold.json'
joblib.dump(best_pipe,model_file,compress=3)
threshold_file.write_text(json.dumps({'threshold':final_tau,'feature_order':numeric_all,
    'positive_label':'fraude/phishing según PhishVN','selection':'validation con FPR<=5%'},
    ensure_ascii=False,indent=2),encoding='utf-8')
restored=joblib.load(model_file)
with threshold_file.open(encoding='utf-8') as handle: cfg=json.load(handle)
assert list(X_test.columns)==cfg['feature_order']
original=best_pipe.predict_proba(X_test.iloc[:1])[:,1][0]
reloaded=restored.predict_proba(X_test.iloc[:1])[:,1][0]
assert np.isclose(original,reloaded)
print('Archivo:',model_file,'| tamaño MB:',round(model_file.stat().st_size/1e6,3))
print('Probabilidad de primera URL de test:',round(float(reloaded),4),
      '| decisión con umbral',cfg['threshold'],':',int(reloaded>=cfg['threshold']))
print('Versiones:',{'python':__import__('sys').version.split()[0],'sklearn':sklearn.__version__,
    'pandas':pd.__version__,'numpy':np.__version__,'plotly':plotly.__version__,
    'joblib':joblib.__version__})

Archivo: c:\UCS\2º Año\1º Cuatrimestre\Data Science\2º Entrega Intermedia\PhishGuard_AI_SEGUNDA_ENTREGA_COMPLETA\project\model.joblib | tamaño MB: 1.539
Probabilidad de primera URL de test: 0.9246 | decisión con umbral 0.81 : 1
Versiones: {'python': '3.13.14', 'sklearn': '1.8.0', 'pandas': '3.0.3', 'numpy': '2.4.6', 'plotly': '7.1.0', 'joblib': '1.5.3'}


*Qué significa para el responsable:* el modelo se recupera; el despliegue todavía requiere nuevas pruebas.

## 13. Executive summary (escrito tras ejecutar)
**Pregunta:** ¿podemos priorizar URLs potencialmente fraudulentas con pocas falsas alarmas? **Solución:** se compararon la regla de longitud, la clase mayoritaria, regresión logística y Random Forest. **Resultado en test:** Random Forest alcanzó **68,0 % de recall**, **2,73 % de FPR**, **97,06 % de precisión** y **F1 = 0,800** con un umbral de **0,81**, seleccionado en validación. **Decisión:** piloto supervisado, **no bloqueo automático**: la meta de recall ≥ 90 % no se cumplió. **Límite:** la etiqueta reúne distintos fraudes, la muestra procede de Vietnam y faltan muchas fechas.

*Qué significa para el responsable:* mejora la priorización, pero necesita supervisión humana.

## 14. References — Referencias (APA 7)

Nguyen Vu, T. (2026). *PhishVN: A time-stamped Vietnamese URL phishing dataset with impersonation-scenario labels and confidence tiers* (versión 4) [Conjunto de datos]. Mendeley Data. https://doi.org/10.17632/b97hxbxtpd.4

Nguyen Vu, T. (2026). *Datasheet, schema y data sources de PhishVN v3.1.0* [Documentación incluida en el ZIP del conjunto de datos]. Mendeley Data. https://doi.org/10.17632/b97hxbxtpd.4

scikit-learn developers. (s. f.). *User guide*. https://scikit-learn.org/stable/user_guide.html

Plotly. (s. f.). *Plotly Python open source graphing library*. https://plotly.com/python/

Creative Commons. (s. f.). *Attribution 4.0 International (CC BY 4.0)*. https://creativecommons.org/licenses/by/4.0/

**Declaración metodológica:** métricas y conclusiones del modelo proceden de las celdas ejecutadas con el CSV derivado del ZIP aportado. La procedencia de etiquetas, problemas de cobertura y naturaleza del split se obtienen del datasheet y schema adjuntos al dataset. No se consultan las URL maliciosas ni se afirma efectividad fuera de esta muestra.